# **Eksperimen Pemodelan Machine Learning - Wine Quality Dataset**

**Nama**: Ersalomo  
**Email Dicoding**: ersalomo@example.com  
**Tanggal**: 2026-05-27  

---

## Deskripsi Proyek

Notebook ini merupakan tahap eksperimen awal dalam pipeline Machine Learning untuk memprediksi **kualitas anggur (wine quality)**. Kami akan:
1. Memperkenalkan dan memuat dataset
2. Melakukan Exploratory Data Analysis (EDA) menyeluruh
3. Melakukan preprocessing data
4. Menyimpan hasil preprocessing untuk tahap modelling

Dataset ini merupakan versi Wine Quality Dataset dengan beberapa modifikasi sintetis untuk simulasi MLOps.

# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Wine Quality Dataset** yang berisi parameter kimia dari anggur merah (*red wine*) beserta label kualitasnya.

### Fitur-fitur Dataset:
| Fitur | Deskripsi | Tipe |
|-------|-----------|------|
| `fixed_acidity` | Kandungan asam tartrat (tetap) | Numerik |
| `volatile_acidity` | Kandungan asam asetat (mudah menguap) | Numerik |
| `citric_acid` | Kandungan asam sitrat (memperkaya rasa) | Numerik |
| `residual_sugar` | Kadar gula tersisa setelah fermentasi | Numerik |
| `chlorides` | Kandungan sodium klorida | Numerik |
| `free_sulfur_dioxide` | SO2 bebas (mencegah oksidasi & mikroba) | Numerik |
| `total_sulfur_dioxide` | Total SO2 (bebas + terikat) | Numerik |
| `density` | Kepadatan anggur (g/cm³) | Numerik |
| `pH` | Tingkat keasaman (0-14) | Numerik |
| `sulphates` | Kandungan potasium sulfat | Numerik |
| `alcohol` | Persentase kadar alkohol | Numerik |
| **`quality`** | **Label kualitas: 0=rendah, 1=tinggi** | **Target (Biner)** |

**Catatan**: Kolom `citric_acid` dan `pH` memiliki beberapa nilai kosong (NaN) yang disengaja untuk simulasi preprocessing.

# **2. Import Library**

Mengimpor library yang dibutuhkan untuk analisis data dan preprocessing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
import os

warnings.filterwarnings('ignore')

# Pengaturan tampilan
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ Semua library berhasil diimpor!')
print(f'   pandas: {pd.__version__}')
print(f'   numpy : {np.__version__}')

# **3. Memuat Dataset**

Memuat dataset mentah `wine_quality_raw.csv` ke dalam DataFrame Pandas.

In [ ]:
# Path dataset mentah (relatif dari direktori preprocessing/)
raw_data_path = '../wine_quality_raw.csv'

if not os.path.exists(raw_data_path):
    raise FileNotFoundError(f'Dataset tidak ditemukan: {raw_data_path}')

df = pd.read_csv(raw_data_path)

print(f'✅ Dataset berhasil dimuat!')
print(f'   Jumlah baris   : {df.shape[0]:,}')
print(f'   Jumlah kolom   : {df.shape[1]}')
print(f'   Ukuran memori  : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print()

# Tampilkan 5 baris pertama
print('--- 5 Baris Pertama Dataset ---')
df.head()

In [ ]:
# Tampilkan 5 baris terakhir
print('--- 5 Baris Terakhir Dataset ---')
df.tail()

# **4. Exploratory Data Analysis (EDA)**

Melakukan analisis mendalam untuk memahami karakteristik dataset sebelum preprocessing.

## 4.1 Informasi Umum dan Statistik Deskriptif

In [ ]:
# Informasi tipe data dan jumlah nilai non-null
print('--- Informasi Dataset (df.info()) ---')
df.info()

In [ ]:
# Statistik deskriptif semua kolom numerik
print('--- Statistik Deskriptif ---')
df.describe().round(4)

## 4.2 Analisis Missing Values dan Duplikasi

In [ ]:
# Analisis nilai kosong (Missing Values)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Jumlah Missing': missing,
    'Persentase (%)': missing_pct
})
missing_df = missing_df[missing_df['Jumlah Missing'] > 0]

if len(missing_df) > 0:
    print('--- Kolom dengan Nilai Kosong ---')
    print(missing_df)
else:
    print('Tidak ada nilai kosong dalam dataset ini.')

print(f'\nTotal missing values: {df.isnull().sum().sum()}')

# Duplikasi
n_dup = df.duplicated().sum()
print(f'Total baris duplikat: {n_dup} ({n_dup/len(df)*100:.2f}%)')

## 4.3 Distribusi Target Variable

In [ ]:
# Distribusi kelas target
print('--- Distribusi Kelas Target (quality) ---')
target_counts = df['quality'].value_counts()
target_pct = df['quality'].value_counts(normalize=True) * 100

print(f'Kelas 0 (Kualitas Rendah): {target_counts.get(0, 0):,} sampel ({target_pct.get(0, 0):.1f}%)')
print(f'Kelas 1 (Kualitas Tinggi): {target_counts.get(1, 0):,} sampel ({target_pct.get(1, 0):.1f}%)')
print(f'Rasio imbalance           : 1:{target_counts.max()/target_counts.min():.1f}')

# Visualisasi distribusi target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(['Kualitas Rendah (0)', 'Kualitas Tinggi (1)'], 
            [target_counts.get(0, 0), target_counts.get(1, 0)],
            color=colors, alpha=0.85, edgecolor='black', linewidth=0.8)
axes[0].set_title('Distribusi Kelas Target', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Jumlah Sampel')
for i, v in enumerate([target_counts.get(0, 0), target_counts.get(1, 0)]):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie([target_counts.get(0, 0), target_counts.get(1, 0)],
           labels=['Kualitas Rendah (0)', 'Kualitas Tinggi (1)'],
           colors=colors, autopct='%1.1f%%', startangle=90,
           explode=(0.05, 0), shadow=True)
axes[1].set_title('Proporsi Kelas Target', fontsize=13, fontweight='bold')

plt.suptitle('Analisis Distribusi Target Variable', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('./target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik distribusi target disimpan.')

## 4.4 Distribusi Setiap Fitur (Histogram)

In [ ]:
# Distribusi semua fitur numerik
feature_cols = [c for c in df.columns if c != 'quality']

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    # Distribusi berdasarkan kelas target
    df[df['quality'] == 0][col].dropna().hist(
        ax=axes[i], bins=30, alpha=0.6, color='#e74c3c', label='Kualitas Rendah'
    )
    df[df['quality'] == 1][col].dropna().hist(
        ax=axes[i], bins=30, alpha=0.6, color='#2ecc71', label='Kualitas Tinggi'
    )
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frekuensi')
    axes[i].legend(fontsize=8)

# Sembunyikan axis kosong
for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi Fitur berdasarkan Kelas Target', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('./feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik distribusi fitur disimpan.')

## 4.5 Analisis Korelasi

In [ ]:
# Heatmap korelasi
corr_matrix = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Heatmap Korelasi Antar Fitur', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('./correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Fitur dengan korelasi tertinggi terhadap target
print('--- Korelasi Fitur terhadap Target (quality) ---')
target_corr = corr_matrix['quality'].drop('quality').sort_values(ascending=False)
print(target_corr.to_string())

## 4.6 Boxplot - Deteksi Outlier

In [ ]:
# Boxplot untuk mendeteksi outlier pada setiap fitur
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    data_by_class = [
        df[df['quality'] == 0][col].dropna().values,
        df[df['quality'] == 1][col].dropna().values
    ]
    bp = axes[i].boxplot(data_by_class, labels=['Rendah', 'Tinggi'],
                         patch_artist=True, notch=False)
    bp['boxes'][0].set_facecolor('#e74c3c')
    bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor('#2ecc71')
    bp['boxes'][1].set_alpha(0.7)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Value')

for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplot Fitur per Kelas Target (Deteksi Outlier)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('./boxplot_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Boxplot disimpan.')

## 4.7 Ringkasan Temuan EDA

Berdasarkan EDA yang telah dilakukan, berikut temuan utama:

| Aspek | Temuan | Tindakan |
|-------|--------|----------|
| **Dimensi** | ~1,600 baris, 12 kolom | - |
| **Missing Values** | `citric_acid` dan `pH` memiliki NaN | Imputasi dengan median |
| **Duplikat** | Terdapat beberapa baris duplikat | Hapus duplikat |
| **Skala Fitur** | Fitur memiliki skala berbeda-beda | StandardScaler |
| **Imbalance** | Kelas tidak seimbang (lebih banyak kualitas rendah) | `class_weight='balanced'` pada model |
| **Korelasi tinggi** | `alcohol` berkorelasi positif tertinggi dengan target | Fitur penting |
| **Outlier** | Beberapa fitur memiliki outlier (SO2, chlorides) | Dipertahankan (RF robust) |

# **5. Data Preprocessing**

Berdasarkan temuan EDA, preprocessing dilakukan dalam 4 tahap:
1. Hapus duplikat
2. Imputasi missing values dengan median
3. Standardisasi fitur dengan StandardScaler
4. Simpan hasil preprocessing

In [ ]:
# ============================================================
# TAHAP 1: Hapus Data Duplikat
# ============================================================
df_clean = df.drop_duplicates().copy()
n_removed = len(df) - len(df_clean)
print(f'✅ Tahap 1 - Hapus Duplikat:')
print(f'   Sebelum : {len(df):,} baris')
print(f'   Dihapus : {n_removed} baris duplikat')
print(f'   Sesudah : {len(df_clean):,} baris')

In [ ]:
# ============================================================
# TAHAP 2: Imputasi Missing Values dengan Median
# ============================================================
print('✅ Tahap 2 - Imputasi Missing Values:')
impute_cols = ['citric_acid', 'pH']
for col in impute_cols:
    if col in df_clean.columns:
        n_missing = df_clean[col].isnull().sum()
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f'   [{col}]: {n_missing} nilai kosong → diisi dengan median {median_val:.4f}')

print(f'\n   Missing values setelah imputasi: {df_clean.isnull().sum().sum()}')

In [ ]:
# ============================================================
# TAHAP 3: Standardisasi Fitur dengan StandardScaler
# ============================================================
print('✅ Tahap 3 - Standardisasi Fitur:')

features = df_clean.drop(columns=['quality'])
target   = df_clean['quality']

print(f'   Fitur yang distandarisasi: {list(features.columns)}')
print(f'   Mean sebelum scaling (contoh - alcohol): {features["alcohol"].mean():.4f}')

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Konversi kembali ke DataFrame dengan nama kolom asli
df_preprocessed = pd.DataFrame(scaled_features, columns=features.columns)
df_preprocessed['quality'] = target.reset_index(drop=True)

print(f'   Mean sesudah scaling (contoh - alcohol): {df_preprocessed["alcohol"].mean():.6f} (~0)')
print(f'   Std sesudah scaling (contoh - alcohol): {df_preprocessed["alcohol"].std():.4f} (~1)')
print(f'\n   Dimensi hasil preprocessing: {df_preprocessed.shape}')
df_preprocessed.describe().round(4)

In [ ]:
# ============================================================
# TAHAP 4: Simpan Hasil Preprocessing
# ============================================================
print('✅ Tahap 4 - Menyimpan Hasil Preprocessing:')

# Direktori output utama
output_dir = './namadataset_preprocessing'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'wine_preprocessed.csv')
df_preprocessed.to_csv(output_path, index=False)
print(f'   Tersimpan: {output_path}')

# Juga simpan ke level preprocessing/ (digunakan oleh automate_siswa.py)
df_preprocessed.to_csv('./wine_preprocessed.csv', index=False)
print(f'   Tersimpan: ./wine_preprocessed.csv')

# Verifikasi file tersimpan
file_size = os.path.getsize(output_path) / 1024
print(f'   Ukuran file: {file_size:.1f} KB')
print(f'   Jumlah baris: {len(df_preprocessed):,}')
print(f'   Distribusi target: {df_preprocessed["quality"].value_counts().to_dict()}')
print('\n🎉 Preprocessing selesai!')

## 5.1 Verifikasi Hasil Preprocessing

In [ ]:
# Verifikasi 5 baris pertama hasil preprocessing
print('--- Hasil Preprocessing (5 baris pertama) ---')
df_preprocessed.head()

In [ ]:
# Perbandingan distribusi sebelum dan sesudah scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sebelum scaling
df_clean[feature_cols].boxplot(ax=axes[0], rot=45)
axes[0].set_title('Distribusi Fitur SEBELUM Scaling', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Nilai')

# Sesudah scaling
df_preprocessed[feature_cols].boxplot(ax=axes[1], rot=45)
axes[1].set_title('Distribusi Fitur SESUDAH StandardScaler', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Nilai (Terstandarisasi)')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.7, label='Mean=0')

plt.suptitle('Perbandingan Distribusi Fitur Sebelum & Sesudah Preprocessing', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('./before_after_scaling.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik perbandingan scaling disimpan.')

# **Kesimpulan**

Eksperimen preprocessing telah berhasil dilakukan dengan:

1. **Dataset dimuat** dari `wine_quality_raw.csv` dengan total ~1,600 baris dan 12 kolom
2. **EDA** mengungkapkan:
   - Terdapat missing values pada `citric_acid` dan `pH`
   - Terdapat beberapa baris duplikat
   - Kelas target tidak seimbang (class imbalance)
   - `alcohol` memiliki korelasi tertinggi dengan kualitas wine
3. **Preprocessing** dilakukan dalam 4 tahap:
   - Hapus duplikat
   - Imputasi missing values dengan median
   - Standardisasi fitur dengan StandardScaler
   - Simpan hasil ke CSV
4. **Dataset siap** untuk tahap modelling di `Membangun_model/`

Dataset hasil preprocessing disimpan di:
- `./namadataset_preprocessing/wine_preprocessed.csv` (untuk referensi)
- `./wine_preprocessed.csv` (untuk digunakan oleh `automate_siswa.py`)